In [17]:
from collections import deque

gudang_barang = {
    "BRG001" : {"NAMA" : "MOUSE GAMING",  "HARGA" : 150000,  "STOK" : 10},
    "BRG002" : {"NAMA" : "KEYBOARD MECH", "HARGA" : 450000,  "STOK" : 5},
    "BRG003" : {"NAMA" : "HEADSET",       "HARGA" : 250000,  "STOK" : 8},
    "BRG004" : {"NAMA" : "MONITOR",       "HARGA" : 2000000, "STOK" : 3},
    "BRG005" : {"NAMA" : "CPU GAMING",    "HARGA" : 5000000, "STOK" : 2},
    "BRG006" : {"NAMA" : "RAM 16GB",      "HARGA" : 800000,  "STOK" : 15},
}

keranjang_belanja = []
riwayat_transaksi = []
antrean_pesanan = deque()


class Node:
    def __init__(self, key, data):
        self.left  = None
        self.right = None
        self.key   = key
        self.data  = data

class BST:
    def __init__(self):
        self.root = None

    def tambah_gudang(self, key, data):
        if self.root is None:
            self.root = Node(key, [data])
        else:
            self._tambah_gudang(self.root, key, data)

    def _tambah_gudang(self, node, key, data):
        if key == node.key:
            node.data.append(data)
        elif key < node.key:
            if node.left is None:
                node.left = Node(key, [data])
            else:
                self._tambah_gudang(node.left, key, data)
        else:
            if node.right is None:
                node.right = Node(key, [data])
            else:
                self._tambah_gudang(node.right, key, data)

    def inorder(self, node, hasil=None):
        if hasil is None:
            hasil = []
        if node:
            self.inorder(node.left, hasil)
            hasil.append((node.key, node.data))
            self.inorder(node.right, hasil)
        return hasil

    def search(self, node, key):
        if node is None:
            return None
        if key == node.key:
            return node.data
        elif key < node.key:
            return self.search(node.left, key)
        else:
            return self.search(node.right, key)


bst_harga = BST()
for id_barang, info in gudang_barang.items():
    bst_harga.tambah_gudang(info["HARGA"], {"id_barang": id_barang, "NAMA": info["NAMA"], "HARGA": info["HARGA"], "STOK": info["STOK"]})


def hitung_total(keranjang, index=0):
    if index == len(keranjang):
        return 0

    item = keranjang[index]
    total_item = gudang_barang[item["id_barang"]]["HARGA"] * item["Jumlah"]
    return total_item + hitung_total(keranjang, index + 1)


def tampilkan_data_barang():
    print("\nDAFTAR BARANG NUSA MART")
    print("ID \t NAMA \t  HARGA \t  STOK")
    for id_barang, info in gudang_barang.items():
        print(id_barang, "\t", info["NAMA"], "\t", info["HARGA"], "\t", info["STOK"])


def tambah_ke_keranjang():
    id_barang = input("Masukkan ID barang: ").upper()
    if id_barang not in gudang_barang:
        print("ID barang tidak ditemukan.")
        return

    barang = gudang_barang[id_barang]
    if barang["STOK"] == 0:
        print("\nStok barang habis.")
        return
    try:
        jumlah = int(input("Masukkan jumlah (stok tersedia: " + str(barang["STOK"]) + "): "))
        if jumlah <= 0:
            print("Jumlah harus lebih dari 0.")
            return
        if jumlah > barang["STOK"]:
            print("Jumlah melebihi stok yang tersedia.")
            return
    except ValueError:
        print("\nInput tidak valid.")
        return

    for item in keranjang_belanja:
        if item["id_barang"] == id_barang:
            if item["Jumlah"] + jumlah > barang["STOK"]:
                print("Jumlah melebihi stok yang tersedia.")
                return
            item["Jumlah"] = item["Jumlah"] + jumlah
            print(barang["NAMA"] + " diupdate. Jumlah: " + str(item["Jumlah"]))
            return

    keranjang_belanja.append({"id_barang": id_barang, "Jumlah": jumlah})
    print("\n" + barang["NAMA"] + " ditambahkan ke keranjang. Jumlah: " + str(jumlah))


def lihat_isi_keranjang():
    if len(keranjang_belanja) == 0:
        print("\nKeranjang kosong.")
        return
    
    print("\nISI KERANJANG")
    print("No \t Nama \t  HARGA \t  Qty \t Subtotal")
    nomor = 1
    for item in keranjang_belanja:
        info_barang = gudang_barang[item["id_barang"]]
        subtotal = info_barang["HARGA"] * item["Jumlah"]
        print(str(nomor), "\t", info_barang["NAMA"], "\t", info_barang["HARGA"], "\t", item["Jumlah"], "\t", subtotal)
        nomor = nomor + 1
    print("\nTOTAL: Rp" + str(hitung_total(keranjang_belanja)))


def tampilan_harga_bst():
    print("\nDAFTAR HARGA BARANG")
    print("HARGA \t NAMA \t ID")
    hasil = bst_harga.inorder(bst_harga.root)
    for harga, barang_list in hasil:
        for b in barang_list:
            print(str(harga), "\t", b["NAMA"], "\t", b["id_barang"])


def checkout():
    if len(keranjang_belanja) == 0:
        print("\nKeranjang kosong.")
        return

    lihat_isi_keranjang()
    konfirmasi = input("Lanjut pembayaran? (y/n): ").lower().strip()
    if konfirmasi != "y":
        print("\nPembayaran dibatalkan.")
        return

    pembayaran = []
    for item in keranjang_belanja:
        pembayaran.append({"id_barang": item["id_barang"], "Jumlah": item["Jumlah"]})
        gudang_barang[item["id_barang"]]["STOK"] = gudang_barang[item["id_barang"]]["STOK"] - item["Jumlah"]

    total_bayar = hitung_total(keranjang_belanja)
    no_transaksi = "TRX" + str(len(riwayat_transaksi) + 1).zfill(4)
    data_transaksi = {"no": no_transaksi, "pembayaran": pembayaran, "total": total_bayar}

    riwayat_transaksi.append(data_transaksi)
    antrean_pesanan.append(data_transaksi)
    keranjang_belanja.clear()
    print("\nPembayaran berhasil! Dengan No. Transaksi: " + no_transaksi)
    print("Total Pembayaran: Rp" + str(total_bayar))


def tampilkan_transaksi():
    if len(riwayat_transaksi) == 0:
        print("\nBelum ada transaksi.")
        return
    print("\nRIWAYAT TRANSAKSI")

    for transaksi in riwayat_transaksi:
        print("No:", transaksi["no"], "| Total: Rp" + str(transaksi["total"]))
        for item in transaksi["pembayaran"]:
            nama  = gudang_barang[item["id_barang"]]["NAMA"]
            harga = gudang_barang[item["id_barang"]]["HARGA"]
            print("  -", nama, "x", item["Jumlah"], "@ Rp" + str(harga))


def undo_transaksi():
    if len(riwayat_transaksi) == 0:
        print("\nTidak ada transaksi yang bisa di-undo.")
        return
    transaksi_terakhir = riwayat_transaksi.pop()
    print("\nMembatalkan transaksi: " + transaksi_terakhir["no"])

    for item in transaksi_terakhir["pembayaran"]:
        nama = gudang_barang[item["id_barang"]]["NAMA"]
        gudang_barang[item["id_barang"]]["STOK"] = gudang_barang[item["id_barang"]]["STOK"] + item["Jumlah"]
        print("Stok " + nama + " dikembalikan: " + str(gudang_barang[item["id_barang"]]["STOK"]))

    antrean_baru = deque()

    for pesanan in antrean_pesanan:
        if pesanan["no"] != transaksi_terakhir["no"]:
            antrean_baru.append(pesanan)
    antrean_pesanan.clear()
    antrean_pesanan.extend(antrean_baru)
    print("\nTransaksi " + transaksi_terakhir["no"] + " berhasil di-undo!")


def tampilkan_antrean():
    if len(antrean_pesanan) == 0:
        print("\nTidak ada pesanan dalam antrean.")
        return
    print("\nANTREAN PESANAN")

    nomor = 1
    for pesanan in antrean_pesanan:
        print("\n" + str(nomor) + ". No:", pesanan["no"], "\t Total: Rp" + str(pesanan["total"]))
        for item in pesanan["pembayaran"]:
            print("   -", gudang_barang[item["id_barang"]]["NAMA"], "x", item["Jumlah"])
        nomor = nomor + 1


def cari_barang():
    try:
        harga_cari = int(input("Masukkan harga barang yang dicari: Rp "))
    except ValueError:
        print("Input tidak valid.")
        return

    hasil = bst_harga.search(bst_harga.root, harga_cari)

    if hasil is None:
        print("\nBarang dengan harga Rp" + str(harga_cari) + " tidak ditemukan.")
    else:
        print("\nBarang ditemukan:")
        for b in hasil:
            stok = gudang_barang[b["id_barang"]]["STOK"]
            print("ID:", b["id_barang"], "\t Nama:", b["NAMA"], "\t Harga: Rp" + str(b["HARGA"]), "\t Stok:", stok)


def main():
    print("Selamat datang di NUSA MART!")
    while True:
        print("\nMenu:")
        print("1. Tampilkan daftar barang")
        print("2. Tambah barang ke keranjang")
        print("3. Lihat isi keranjang")
        print("4. Tampilkan harga barang")
        print("5. Checkout")
        print("6. Tampilkan transaksi")
        print("7. Undo transaksi")
        print("8. Tampilkan antrean pesanan")
        print("9. Cari barang")
        print("10. Keluar")
        pilihan = input("Pilih menu (1-10): ").strip()
        if pilihan == "1":
            tampilkan_data_barang()
        elif pilihan == "2":
            tambah_ke_keranjang()
        elif pilihan == "3":
            lihat_isi_keranjang()
        elif pilihan == "4":
            tampilan_harga_bst()
        elif pilihan == "5":
            checkout()
        elif pilihan == "6":
            tampilkan_transaksi()
        elif pilihan == "7":
            undo_transaksi()
        elif pilihan == "8":
            tampilkan_antrean()
        elif pilihan == "9":
            cari_barang()
        elif pilihan == "10":
            print("Terima kasih telah berbelanja di NUSA MART!")
            break
        else:
            print("Pilihan tidak valid.")

if __name__ == "__main__":
    main()

Selamat datang di NUSA MART!

Menu:
1. Tampilkan daftar barang
2. Tambah barang ke keranjang
3. Lihat isi keranjang
4. Tampilkan harga barang
5. Checkout
6. Tampilkan transaksi
7. Undo transaksi
8. Tampilkan antrean pesanan
9. Cari barang
10. Keluar

MOUSE GAMING ditambahkan ke keranjang. Jumlah: 2

Menu:
1. Tampilkan daftar barang
2. Tambah barang ke keranjang
3. Lihat isi keranjang
4. Tampilkan harga barang
5. Checkout
6. Tampilkan transaksi
7. Undo transaksi
8. Tampilkan antrean pesanan
9. Cari barang
10. Keluar

KEYBOARD MECH ditambahkan ke keranjang. Jumlah: 1

Menu:
1. Tampilkan daftar barang
2. Tambah barang ke keranjang
3. Lihat isi keranjang
4. Tampilkan harga barang
5. Checkout
6. Tampilkan transaksi
7. Undo transaksi
8. Tampilkan antrean pesanan
9. Cari barang
10. Keluar

ISI KERANJANG
No 	 Nama 	  HARGA 	  Qty 	 Subtotal
1 	 MOUSE GAMING 	 150000 	 2 	 300000
2 	 KEYBOARD MECH 	 450000 	 1 	 450000

TOTAL: Rp750000

Pembayaran berhasil! Dengan No. Transaksi: TRX0001
Total P